# Defining Feature Columns and Target Variables

Before you train any supervised learning model, you must answer one foundational question with absolute clarity:

**What exactly am I predicting, and what information am I allowed to use to predict it?**

This question is not philosophical. It is operational. The answer determines whether your model is valid, whether your evaluation is honest, and whether your system will work in production.

## Overview

This lesson focuses on defining feature columns (X) and the target variable (y) correctly — a step that appears deceptively simple but determines whether your entire pipeline is valid, leak-free, and aligned with the real-world problem you're trying to solve.

Many ML failures do not happen because of bad algorithms or insufficient data. They happen because:

- The target was defined incorrectly
- Features accidentally included information that would not exist at prediction time
- The separation between features and target was not enforced rigorously

**The discipline you develop here — the habit of asking "would I actually have this information at prediction time?" — is what separates practitioners who build reliable systems from those who build impressive notebooks that fail on deployment.**

## Understanding the Target Variable

The target variable is the column your model is trying to predict. It represents the business outcome of interest. In supervised learning, this is the "correct answer" provided during training — the ground truth that supervises the learning process.

### Examples

| Business Problem | Target Column | Type | Values |
|-----------------|--------------|------|--------|
| Customer churn prediction | Churn | Binary classification | Yes, No |
| House price estimation | SalePrice | Regression | 235,000, ... |
| Fraud detection | IsFraud | Binary classification | 0 (Legitimate), 1 (Fraud) |
| Delivery time estimation | DeliveryMinutes | Regression | 25, 37, 42, ... |
| Disease diagnosis | DiagnosisCode | Multi-class classification | Healthy, Type1, Type2, ... |

## The Three Conditions Every Target Must Satisfy

A valid target variable must meet three non-negotiable criteria:

### 1. It must be clearly defined and measurable

Vague targets lead to vague models. "Customer satisfaction" is not a valid target unless you can point to a specific column with specific values. "Satisfaction score (1-5)" is valid. "Probably satisfied" is not.

### 2. It must be available in historical data

You cannot train a supervised model without labeled examples. If you want to predict customer lifetime value but have no historical data on actual lifetime value, you cannot build a supervised model.

### 3. It must represent a real decision or outcome the business cares about

The target defines success. If you optimize for the wrong target, you build a system that solves the wrong problem.

## Before Writing Any Code, State Clearly:

1. What column is the target?
2. What does each value in that column represent in business terms?
3. Is it classification or regression?
4. If classification: binary, multi-class, or multi-label?
5. If regression: bounded or unbounded, continuous or count-based?
6. What does a correct prediction mean for this business?

**Clarity here prevents confusion later. When evaluation results are unclear or model behavior is unexpected, the first place to look is the target definition. If the target doesn't represent what you actually care about, no amount of feature engineering will fix it.**

## Understanding Feature Columns

Feature columns are the input variables used to predict the target. These are the signals the model learns from — the information available at prediction time that contains predictive patterns about the outcome.

### Example: Customer Churn Features

| Feature | Description | Why It's Valid |
|---------|-------------|----------------|
| Tenure | Number of months as customer | Known at prediction time |
| MonthlyCharges | Current monthly bill | Known at prediction time |
| ContractType | Month-to-month, annual, etc. | Known at prediction time |
| PaymentMethod | Credit card, bank transfer, etc. | Known at prediction time |
| TotalCharges | Cumulative charges to date | Known at prediction time |
| SupportCallsLast90Days | Recent support interactions | Known at prediction time |

Each of these features represents information that exists before you need to make the churn prediction.

## The Golden Rule of Feature Validity

### ✅ Features should represent information that is available at prediction time.

**This is not negotiable. This is the foundational principle that separates valid ML systems from broken ones.**

### Example of Feature Leakage (Invalid)

Including `CancellationReason` in a churn prediction model is invalid. That information only exists after churn has occurred. At the moment you need to predict churn — before it happens — this feature does not exist. A model trained with this feature will perform brilliantly in training and fail completely in production.

### Example of Valid Feature

Including `NumberOfSupportCallsLast90Days` is valid. This represents historical behavior that has already occurred and is available in your customer database at the moment you need to make the prediction.

### The Test

**Close your eyes and imagine the moment in the real world when you need to make a prediction. What information do you have at that exact moment? Those are your valid features. Everything else is leakage.**

## Feature Selection Must Be Intentional, Not Automatic

A common beginner mistake:

```python
# WRONG: Use everything except the target
X = df.drop(columns=['Churn'])  # 49 columns as features
```

This is lazy and dangerous. Not every column should be a feature.

### For Each Candidate Feature, Ask:

1. ✅ Is this available at prediction time?
2. ✅ Does this represent information I'm allowed to use?
3. ❌ Does this contain signal about the target, or is it noise?
4. ❌ Is this derived from the target in a way that creates leakage?

**Accept a column as a feature only after answering yes to the first two questions and no to the last two.**

## Separating X and y in Code

In practice, feature and target separation must be explicit and happen early in the pipeline.

### The Standard Convention

- **X** (uppercase) = feature matrix (all input columns)
- **y** (lowercase) = target vector (the column you're predicting)

### Basic Separation

In [ ]:
import pandas as pd

# Define target explicitly
TARGET_COLUMN = "Churn"

# Load data
df = pd.read_csv('data/churn.csv')

# Separate features and target
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print(f"Features shape: {X.shape}")  # (7043, 19)
print(f"Target shape: {y.shape}")    # (7043,)

## More Rigorous Separation with Explicit Feature Lists

In [ ]:
# Define features explicitly in config
NUMERICAL_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_FEATURES = ['gender', 'Contract', 'PaymentMethod', 'InternetService']
TARGET_COLUMN = 'Churn'
EXCLUDED_COLUMNS = ['CustomerID']

# Combine feature lists
ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

# Separate
X = df[ALL_FEATURES]
y = df[TARGET_COLUMN]

# Verify target is not in features
assert TARGET_COLUMN not in X.columns, "Target leaked into features!"

print(f"✓ Features separated successfully")
print(f"  - Numerical: {len(NUMERICAL_FEATURES)}")
print(f"  - Categorical: {len(CATEGORICAL_FEATURES)}")
print(f"  - Total: {len(ALL_FEATURES)}")

### Key Benefits of Explicit Approach

- Prevents accidental inclusion of the target
- Prevents accidental inclusion of excluded columns
- Makes the feature definition reviewable and reproducible
- Makes code modular (feature lists imported from config, not scattered)

## Feature Eligibility Criteria

Not every column in a dataset should automatically become a feature. Evaluate each column carefully.

### A Valid Feature Should:

✅ Be available at prediction time

✅ Be relevant to the target

✅ Not directly encode the target

✅ Not be a post-outcome variable

✅ Have acceptable data quality

## Categories of Columns to Treat Cautiously

### 1. IDs and Unique Identifiers

**Examples:** CustomerID, TransactionID, OrderID, SessionID

**Problem:** These columns uniquely identify each row. They carry no generalizable predictive signal. A model that memorizes specific customer IDs from the training set learns nothing useful for predicting on new customers.

**Decision:** Almost always exclude unless you can justify why the ID itself contains predictive signal.

### 2. Raw Timestamps

**Examples:** OrderDate, TransactionTimestamp, SignupDate

**Problem:** Raw timestamps are not useful as features. The model cannot learn from "2024-03-15 14:23:17" in a way that generalizes.

**Solution:** Transform into derived features:
- DayOfWeek — Monday = 1, Tuesday = 2, etc.
- HourOfDay — 0-23
- IsWeekend — binary flag
- DaysSinceSignup — recency measure

**Decision:** Exclude raw timestamps. Include derived temporal features.

### 3. Free-Text Descriptions

**Examples:** CustomerComments, ProductDescription, EmailBody

**Problem:** Text cannot be used directly as a feature. Models require numerical inputs.

**Solution:** Transform text:
- TF-IDF vectors
- Word embeddings
- Sentiment scores
- Text length

**Decision:** Exclude raw text. Include text-derived numerical features.

### 4. Target-Derived Columns (Leakage) ⚠️

**Examples:**
- DaysUntilChurn (calculated from churn date)
- DefaultAmount (only known if default occurred)
- FraudInvestigationOpened (only after fraud suspected)

**Problem:** These columns are calculated from or only exist because the target outcome occurred. They are perfect predictors in training and completely unavailable in production.

**Detection:** If correlation with target > 0.95, investigate for leakage.

**Decision:** Always exclude target-derived columns.

### 5. Post-Event Data

**Example:** Predicting hospital readmission and including FollowUpAppointmentBooked if that appointment is scheduled after discharge.

**Problem:** This creates circular dependency. The feature is generated based on the prediction the model is supposed to make.

**Decision:** Exclude any information generated after the prediction decision point.

## The Core Question: Would This Feature Exist at Prediction Time?

### For every candidate feature, ask:

**"At the exact moment in the real world when I need to make a prediction, would I have access to this information?"**

- If the answer is **no** or **uncertain** → exclude the feature
- If the answer is **yes** → include the feature

**This question cuts through ambiguity and prevents leakage by design.**

## Handling Categorical vs Numerical Features

After identifying valid feature columns, categorize them by data type. This determines how they will be preprocessed.

### Numerical Features

- Continuous values: Age, Salary, MonthlyCharges, Temperature
- Integer counts: NumberOfPurchases, YearsExperience, Tenure
- Can be used directly by most models
- May require scaling or normalization

### Categorical Features

- Discrete labels with no intrinsic ordering: Gender, ContractType, PaymentMethod, Region
- Must be encoded (one-hot, label encoding, target encoding) before use

### Ordinal Features (Special Case)

- Discrete labels with natural ordering: EducationLevel, SatisfactionRating
- Can be label-encoded with meaningful integer mappings

**Failing to distinguish them leads to preprocessing errors.**

## Explicit Configuration Example

In [ ]:
# config.py

TARGET_COLUMN = 'Churn'

NUMERICAL_FEATURES = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

CATEGORICAL_FEATURES = [
    'gender',
    'Contract',
    'PaymentMethod',
    'InternetService',
    'OnlineSecurity',
    'TechSupport'
]

EXCLUDED_COLUMNS = [
    'CustomerID'  # Identifier, no predictive value
]

# Validation
ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
assert TARGET_COLUMN not in ALL_FEATURES, "Target in features!"

print(f"✓ Configuration valid")
print(f"  - Numerical: {len(NUMERICAL_FEATURES)}")
print(f"  - Categorical: {len(CATEGORICAL_FEATURES)}")
print(f"  - Total: {len(ALL_FEATURES)}")

## Avoiding Data Leakage During Feature Definition

Data leakage is the most dangerous and most common mistake in supervised learning. It happens when the model sees information during training that it would not have access to in production.

### Leakage Produces Models That:

- Achieve 95%+ accuracy in training ✓
- Achieve 95%+ accuracy in cross-validation ✓
- Achieve 50% accuracy in production ✗

Because the evaluation was contaminated, you never discovered the problem until deployment.

## Common Sources of Data Leakage

### 1. Target Leakage

Including features derived from the target or only existing because the target is known.

**Example:** Including ChurnDate in a churn prediction model. You don't know the churn date until after churn happens.

### 2. Temporal Leakage

Including features that use information from the future.

**Example:** Predicting quarterly sales and including NextQuarterReturns.

### 3. Train/Test Contamination

Fitting preprocessing transformations on all data before splitting, causing information from the test set to leak into training.

**Incorrect:**
```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Fit on ALL data including test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

**Correct:**
```python
X_train, X_test, y_train, y_test = train_test_split(X, y)  # Split FIRST
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on train only
X_test_scaled = scaler.transform(X_test)      # Transform test
```

### 4. Post-Processing Leakage

Using the target or evaluation results to adjust features retroactively.

**Example:** After seeing poor performance on high-value customers, adding a feature IsHighValue defined as "top 10% of target variable."

## The Correct Sequence to Prevent Leakage

1. Load data
2. Define target and features explicitly (this lesson)
3. Remove excluded columns
4. **Separate X and y** ← This is where leakage is prevented
5. **Split into train/test** ← Before any fitting
6. Fit preprocessing on training data only
7. Apply preprocessing to test data
8. Train model
9. Evaluate on test data

**If feature definition is sloppy — if you don't rigorously separate features from target, if you don't think carefully about temporal availability, if you don't split before fitting — leakage becomes invisible until production.**

## Documenting Feature and Target Definitions

For professional ML projects, feature and target definitions must be documented clearly. This is not optional. It is a professional responsibility.

### What to Document

1. **Target column name and description**
   - What it represents
   - What each value means
   - Classification or regression

2. **Feature column list**
   - All included features
   - Brief description of each
   - Why each is valid

3. **Excluded columns and reasons**
   - Which columns are not used
   - Why (identifier, leakage, irrelevant, etc.)

4. **Problem type**
   - Binary classification, multi-class, regression, etc.

5. **Evaluation metrics**
   - What metrics will measure success
   - Why those metrics are appropriate

## Real-World Example: Customer Churn Prediction

### Problem
Predict whether a customer will cancel their subscription in the next 30 days.

### Target
- **Column:** Churn
- **Type:** Binary classification
- **Values:** Yes (will churn), No (will not churn)

### Features (Valid — Available at Prediction Time)
- Tenure — number of months as customer
- ContractType — month-to-month, annual, etc.
- MonthlyCharges — current monthly bill
- TotalCharges — cumulative charges to date
- PaymentMethod — how customer pays
- SupportCallsLast90Days — recent support interactions

### Excluded Columns (Invalid — Not Available at Prediction Time)
- CustomerID — unique identifier, no predictive value
- CancellationDate — only exists after churn (temporal leakage)
- CancellationReason — only exists after churn (target leakage)

### Validation
Every feature passes: "Do I have this information 30 days before potential churn?" YES
Every excluded column fails: "Do I have this information at prediction time?" NO

## Common Mistakes in Defining Features and Targets

### Mistake 1: Including the Target as a Feature

```python
# WRONG: Only drops ID, keeps target in features
X = df.drop(columns=['CustomerID'])
y = df['Churn']
```

Now Churn appears in both X and y. The model achieves 100% accuracy by copying Churn from X.

**Correct:**
```python
# RIGHT: Drop both ID and target
X = df.drop(columns=['CustomerID', 'Churn'])
y = df['Churn']
```

### Mistake 2: Using Future Information

In loan default prediction, including DaysUntilDefault is invalid (you don't know until after default).

**Correct:** Use only information at loan origination: CreditScore, Income, DebtToIncomeRatio.

### Mistake 3: Not Removing Identifiers

The model memorizes IDs from training, which is useless for new customers.

**Correct:** Always exclude CustomerID, TransactionID, etc.

### Mistake 4: Leaking Through Derived Features

```python
# WRONG: Includes current customer's churn status
df['ChurnRate'] = df.groupby('Region')['Churn'].transform('mean')
```

**Correct:** Compute historical rate excluding current customer

## Best Practices for Defining Features and Targets

### Practice 1: Centralize Configuration

Define features and target in a central config file so every module uses identical definitions.

### Practice 2: Validate Feature Definitions

Write validation functions that check for common errors:
- Target not in features
- No ID-like columns in features
- No suspiciously high correlations (leakage detection)

### Practice 3: Document Feature Availability

For each feature, document when it becomes available in the business process.

### Practice 4: Simulate the Prediction Scenario

Before finalizing features, mentally simulate the real prediction:

**Scenario:** "It is February 15, 2026. I need to predict which customers will churn by March 15."

Ask yourself:
- What information exists in my database on February 15?
- What information does not exist yet?
- What information only exists after churn happens?

## Closing Reflection: Features and Targets Define the Problem

Before you preprocess data, before you train models, before you evaluate results, you must clearly and correctly define your features and target.

This definition determines:
- What the model can learn from
- Whether the model represents a realistic prediction scenario
- Whether evaluation results are meaningful
- Whether the system will work in production
- Whether the entire project succeeds or fails

### Ask Yourself These Questions:

1. Have I clearly identified the target variable — the one thing I'm predicting?
2. Can I explain in business terms what each value of the target represents?
3. Have I identified all legitimate features — information available at prediction time?
4. Have I removed all forms of data leakage?
5. Have I removed metadata and identifiers?
6. Can I explain why each feature is legitimate?
7. Would I have each feature at the exact moment I need to make a prediction?

**If the answer to any of these is unclear, stop and clarify before proceeding.**

A model trained on leaked features produces impressive training metrics and disastrous real-world results. A model missing critical features never achieves acceptable performance.

**Define features and targets correctly. Everything else depends on this foundation.**

## Bonus Content 🎁

This section is optional. Learners who want to deepen their understanding of feature definition, target selection, and data leakage prevention can explore these resources:

- Feature Engineering and Selection - Max Kuhn and Kjell Johnson (Free Online)
- Data Leakage in Machine Learning - Kaggle
- Leakage in Data Mining: Formulation, Detection, and Avoidance - Kaufman et al.
- Feature Engineering for Machine Learning - Alice Zheng and Amanda Casari
- Google ML Guide - Data Preparation and Feature Engineering
- Scikit-learn Feature Selection Guide